In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, GridSearchCV



def load_awards_players_data():
    try:
        awards_players = pd.read_csv('../dataset/awards_players.csv')
        return awards_players

    except FileNotFoundError:
        print("Error: awards_players.csv file not foind in dataset folder!")
        return None

    except Exception as e:
        print(f"Error loading data: {e}")
        return None


def load_coaches_data():
    try:
        coaches = pd.read_csv('../dataset/coaches.csv')
        return coaches

    except FileNotFoundError:
        print("Error: coaches.csv file not foind in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


def load_players_teams_data():
    try:
        players_teams = pd.read_csv('../dataset/players_teams.csv')
        return players_teams

    except FileNotFoundError:
        print("Error: players_teams.csv file not foind in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data {e}")
        return None
    

def load_players_data():
    try:
        players = pd.read_csv('../dataset/players.csv')
        return players

    except FileNotFoundError:
        print("Error: players.csv file not foind in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data {e}")
        return None
    

def load_series_data():
    try:
        series = pd.read_csv('../dataset/series_post.csv')
        
        return series
        
    except FileNotFoundError:
        print("Error: series.csv file not found in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data: {e}")
        return None   
    

def load_teams_post_data():
    try:
        teams_post = pd.read_csv('../dataset/teams_post.csv')
        
        return teams_post
        
    except FileNotFoundError:
        print("Error: teams_post.csv file not found in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


def load_teams_data():
    try:
        teams = pd.read_csv('../dataset/teams.csv')
        
        return teams
        
    except FileNotFoundError:
        print("Error: teams.csv file not found in dataset folder!")
        return None
    
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

In [ ]:
awards_players = load_awards_players_data()
coaches = load_coaches_data()
players_teams = load_players_teams_data()
players = load_players_data()
series_data = load_series_data()
teams_post_data = load_teams_post_data()
teams_data = load_teams_data()


In [ ]:
team_to_franch_mapping = teams_data.set_index('tmID')['franchID'].to_dict()
print(team_to_franch_mapping)

players_teams['tmID'] = players_teams['tmID'].map(team_to_franch_mapping)

coaches['tmID'] = coaches['tmID'].map(team_to_franch_mapping)

teams_data['tmID'] = teams_data['tmID'].map(team_to_franch_mapping)
teams_data.head(134)

In [ ]:
numeric_teams_data = teams_data.select_dtypes(include=['number'])

correlation = numeric_teams_data.corr()['rank'].drop('rank').sort_values(ascending=False)

print(correlation)

plt.figure(figsize=(10, 12))
plt.barh(correlation.index, correlation.values)
plt.title("Correlation of Each Feature with Rank")
plt.xlabel("Correlation coefficient (Pearson)")
plt.ylabel("Feature")
plt.gca().invert_yaxis()  # Highest correlation at the top
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_teams_data.corr()[['rank']].sort_values(by='rank', ascending=False), 
            annot=True, cmap='coolwarm', center=0)
plt.title("Feature Correlations with Rank (Heatmap)")
plt.show()

In [ ]:
agg_players = players_teams.groupby(['tmID', 'year']).agg({
    "GP": 'sum',
    "GS": 'sum',
    "minutes": 'sum',
    "points": 'sum',
    "oRebounds": 'sum',
    "dRebounds": 'sum',
    "rebounds": 'sum',
    "assists": 'sum',
    "steals": 'sum',
    "blocks": 'sum',
    "turnovers": 'sum',
    "PF": 'sum',
    "fgAttempted": 'sum',
    "fgMade": 'sum',
    "ftAttempted": 'sum',
    "ftMade": 'sum',
    "threeAttempted": 'sum',
    "threeMade": 'sum',
    "dq": 'sum',
    "PostGP": 'sum',
    "PostGS": 'sum',
    "PostMinutes": 'sum',
    "PostPoints": 'sum',
    "PostoRebounds": 'sum',
    "PostdRebounds": 'sum',
    "PostRebounds": 'sum',
    "PostAssists": 'sum',
    "PostSteals": 'sum',
    "PostBlocks": 'sum',
    "PostTurnovers": 'sum',
    "PostPF": 'sum',
    "PostfgAttempted": 'sum',
    "PostfgMade": 'sum',
    "PostftAttempted": 'sum',
    "PostftMade": 'sum',
    "PostthreeAttempted": 'sum',
    "PostthreeMade": 'sum',
    "PostDQ": 'sum'
}).reset_index()

merged = pd.merge(agg_players, teams_data[['tmID', 'year', 'rank']], on=['tmID', 'year'], how='inner')

corr = merged.corr(numeric_only=True)['rank'].drop('rank').sort_values(ascending=False)

print("Correlation of player-level aggregated features with team rank:")
print(corr)

# === 5. Plot correlation ===
plt.figure(figsize=(10, 12))
plt.barh(corr.index, corr.values)
plt.title("Correlation Between Aggregated Player Stats and Team Rank")
plt.xlabel("Correlation (Pearson)")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
team_features = ["o_fgm", "o_fga", "o_ftm", "o_fta", "o_3pm", "o_3pa", "o_oreb", "o_dreb", "o_reb", "o_asts", "o_pf", "o_stl", "o_to",
                "o_blk", "o_pts", "d_fgm", "d_fga", "d_ftm", "d_fta", "d_3pm", "d_3pa", "d_oreb", "d_dreb", "d_reb", "d_asts", "d_pf",
                "d_stl", "d_to", "d_blk", "d_pts", "won", "lost", "GP", "homeW", "homeL", "awayW", "awayL", "confW", "confL"]

teams_subset = teams_data[['tmID', 'year', 'rank'] + team_features]

merged = pd.merge(teams_subset, agg_players, on=['tmID', 'year'], how='inner')

corr = merged.corr(numeric_only=True)['rank'].drop('rank').sort_values(key=abs, ascending=False)
print("\nTop correlated features with rank:")
print(corr.head(40))

top_features = corr.head(26).index.to_list()
print("\nSelected features for training:", top_features)

selected_features = corr[(corr.abs() > 0.4)].index.to_list()
print(f"Selected {len(selected_features)} features with |correlation| > 0.4:")
print(selected_features)


train = merged[merged['year'] < 10]

test = merged[merged['year'] == 10]

X_train = train[selected_features]
y_train = train['rank']
X_test = test[selected_features]
y_test = test['rank']

model = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_split=10, min_samples_leaf=1, random_state=42)
model.fit(X_train, y_train)

# === 8. Predict rankings for year 10 ===
y_pred = model.predict(X_test)
test['predicted_rank'] = y_pred

# === 9. Evaluate ===
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Performance for Year 10:")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R² Score: {r2:.2f}")

# Show predicted vs actual
comparison = test[['tmID', 'year', 'rank', 'predicted_rank']].sort_values(by='predicted_rank')
print("\nPredicted vs Actual Rankings for Year 10:")
print(comparison.to_string(index=False))

# === 10. Visualize predicted vs actual rank ===
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Rank")
plt.ylabel("Predicted Rank")
plt.title("Predicted vs Actual Team Rank (Year 10)")
plt.tight_layout()
plt.show()

importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
plt.figure(figsize=(8,6))
plt.barh(importances.index, importances.values)
plt.title("Feature Importance in Rank Prediction")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

y_train_pred = model.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print(f"Train MAE: {mae_train:.2f}, Train R²: {r2_train:.2f}")

scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
print("Cross-validated R²:", scores.mean(), "±", scores.std())

'''
param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best cross-validated R²:", grid.best_score_)
'''
